In [3]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path(".")

In [4]:
add_to_cart = pd.read_parquet(
    DATA_DIR / "add_to_cart_subsampled.parquet"
)

product_buy = pd.read_parquet(
    DATA_DIR / "product_buy_subsampled.parquet"
)

remove_from_cart = pd.read_parquet(
    DATA_DIR / "remove_from_cart_subsampled.parquet"
)

print("Add-to-cart:", add_to_cart.shape)
print("Product-buy:", product_buy.shape)
print("Remove-from-cart:", remove_from_cart.shape)

Add-to-cart: (4890231, 3)
Product-buy: (1171501, 3)
Remove-from-cart: (2229315, 3)


In [5]:
all_users = set(add_to_cart["client_id"]) \
          | set(product_buy["client_id"]) \
          | set(remove_from_cart["client_id"])

print("Unique active users:", len(all_users))

Unique active users: 520417


In [6]:
add_events = add_to_cart[
    ["client_id", "timestamp"]
].copy()
add_events["event_type"] = "add_to_cart"

buy_events = product_buy[
    ["client_id", "timestamp"]
].copy()
buy_events["event_type"] = "product_buy"

remove_events = remove_from_cart[
    ["client_id", "timestamp"]
].copy()
remove_events["event_type"] = "remove_from_cart"

events = pd.concat(
    [add_events, buy_events, remove_events],
    ignore_index=True
)

events["timestamp"] = pd.to_datetime(events["timestamp"])

events = events.sort_values(
    ["client_id", "timestamp"]
).reset_index(drop=True)

print("Total events:", len(events))
print("Unique users:", events["client_id"].nunique())

events.head()

Total events: 8291047
Unique users: 520417


,client_id,timestamp,event_type
0,33,2022-07-26 16:20:00,add_to_cart
1,33,2022-07-26 16:22:45,product_buy
2,33,2022-07-26 16:25:20,add_to_cart
3,33,2022-07-26 16:25:40,product_buy
4,33,2022-11-28 01:25:55,add_to_cart


In [7]:
sequence_length = (
    events
    .groupby("client_id")
    .size()
    .rename("sequence_length")
)

sequence_length.describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

count    520417.000000
mean         15.931545
std          31.192640
min           5.000000
25%           6.000000
50%           9.000000
75%          15.000000
90%          30.000000
95%          47.000000
99%         118.000000
max        3801.000000
Name: sequence_length, dtype: float64

In [8]:
print("Number of users:", len(sequence_length))
print("Mean sequence length:", round(sequence_length.mean(), 2))
print("Median sequence length:", sequence_length.median())
print("75th percentile:", sequence_length.quantile(0.75))
print("90th percentile:", sequence_length.quantile(0.90))
print("95th percentile:", sequence_length.quantile(0.95))
print("99th percentile:", sequence_length.quantile(0.99))
print("Maximum sequence length:", sequence_length.max())

Number of users: 520417
Mean sequence length: 15.93
Median sequence length: 9.0
75th percentile: 15.0
90th percentile: 30.0
95th percentile: 47.0
99th percentile: 118.0
Maximum sequence length: 3801


In [9]:
anchor_time = events.groupby("client_id")["timestamp"].max()

events_window = events.copy()

events_window["anchor_time"] = (
    events_window["client_id"].map(anchor_time)
)

events_window["days_before_anchor"] = (
    events_window["anchor_time"] - events_window["timestamp"]
).dt.total_seconds() / 86400

In [10]:
window_summary = pd.DataFrame(index=anchor_time.index)

for days in [7, 14, 30]:
    counts = (
        events_window[
            events_window["days_before_anchor"] <= days
        ]
        .groupby("client_id")
        .size()
    )

    window_summary[f"{days}d"] = counts

window_summary = window_summary.fillna(0).astype(int)

window_summary.head()

,7d,14d,30d
client_id,,,
33,2,2,2
80,12,12,12
84,4,8,10
105,12,12,18
131,5,5,5


In [11]:
window_stats = []

for col in ["7d", "14d", "30d"]:
    x = window_summary[col]

    window_stats.append({
        "Window": col,
        "Mean": x.mean(),
        "Median": x.median(),
        "P75": x.quantile(0.75),
        "P90": x.quantile(0.90),
        "P95": x.quantile(0.95),
        "P99": x.quantile(0.99),
        "Max": x.max()
    })

window_stats_df = pd.DataFrame(window_stats)
window_stats_df.round(2)

,Window,Mean,Median,P75,P90,P95,P99,Max
0,7d,7.40,5.0,8.0,14.0,20.0,42.0,1342
1,14d,8.53,6.0,9.0,16.0,24.0,51.0,1342
2,30d,10.36,7.0,11.0,20.0,29.0,65.0,1500
